In [41]:
import tkinter as tk
from tkinter import messagebox
from z3 import Bool, And, Or, Not, Solver, is_true, sat

# Question

* Take a random graph 𝐺 as an input
* Visualize the graph
* Check and find a 4-color solution to the graph coloring problem for 𝐺 using SAT
* Visualize the solution graph
* Are there any condition which if removed yields a trivial solution to the problem?

## Graph

In [42]:
class Graph:
    def __init__(self):
        self.vertices = set()
        self.edges = []
        self.positions = {}

    def add_vertex(self, vertex, x, y):
        self.vertices.add(vertex)
        self.positions[vertex] = (x, y)

    def add_edge(self, vertex1, vertex2):
        self.edges.append((vertex1, vertex2))

    def clear(self):
        self.vertices.clear()
        self.edges.clear()
        self.positions.clear()

## Graph Solver

In [43]:
def solve_graph(graph, colors = None):
    if colors is None:
        colors = ['r', 'g', 'b', 'y']  # Default colors

    s = Solver()

    # Define boolean variables for each vertex and color

    vertex_color_variables = {}
    vertices = graph.vertices
    edges = graph.edges

    for v in vertices:
        for c in colors:
            vertex_color_variables[(v, c)] = Bool(f"V_{v}_{c}")

    for v in vertices:
        s.add(
            Or(
                *(vertex_color_variables[(v, c)] for c in colors)
            )
        ) # Each vertex has at least one color


        s.add(
            And(
                *(
                    Or(
                        Not(vertex_color_variables[(v, c1)]), 
                        Not(vertex_color_variables[(v, c2)])
                    ) 
                    for i, c1 in enumerate(colors) 
                    for j, c2 in enumerate(colors) 
                    if i < j
                )
            )
        ) # Each vertex has only one color


    for u, v in edges:
        s.add(
            And(
                *(
                    Or(
                        Not(vertex_color_variables[(u, c)]), 
                        Not(vertex_color_variables[(v, c)])
                    ) for c in colors
                )
            )
        ) # connected nodes do not have the same color


    if s.check() == sat:
        model = s.model()
        print("Solution found:")
        print(s.model())
        return {
            v: c for v in graph.vertices 
            for c in colors 
            if is_true(model.evaluate(vertex_color_variables[(v, c)]))
        }
    else:
        return None

## GUI

In [44]:
class GraphGUI:

    def __init__(self, solver, attempt_5_colors=False):
        self.root = tk.Tk()
        self.root.title("Color Graph")
        self.root.geometry("1000x800")

        # Graph data
        self.graph = Graph()
        self.solver = solver
        self.attempt_5_colors = attempt_5_colors

        # GUI settings
        self.vertex_radius = 20
        self.min_vertex_distance = 30
        self.next_vertex_id = 0
        self.selected_vertex = None
        self.vertex_items = {}

        # Canvas
        self.canvas = tk.Canvas(
            self.root,
            width=1000,
            height=700,
            bg="white"
        )
        self.canvas.pack()

        # Buttons
        self.button_frame = tk.Frame(self.root)
        self.button_frame.pack(pady=10)

        self.solve_button = tk.Button(
            self.button_frame,
            text="Color Graph",
            command=self.solve
        )

        self.solve_button.pack(side=tk.LEFT, padx=5)

        self.clear_button = tk.Button(
            self.button_frame,
            text="Clear",
            command=self.clear_graph
        )
        self.clear_button.pack(side=tk.LEFT, padx=5)

        # Mouse interaction
        self.canvas.bind("<Button-1>", self.canvas_click)

    def canvas_click(self, event):
        vertex = self.get_vertex_at(event.x, event.y)

        if vertex is None:
            self.create_vertex(event.x, event.y)
        else:
            self.select_vertex(vertex)

    def create_vertex(self, x, y):
        if self.is_position_occupied(x, y):
            return

        vertex = chr(ord("a") + self.next_vertex_id)

        self.next_vertex_id += 1

        self.graph.add_vertex(vertex, x, y)

        self.draw_vertex(vertex)

    def draw_vertex(self, vertex):
        x, y = self.graph.positions[vertex]
        r = self.vertex_radius

        circle = self.canvas.create_oval(
            x - r,
            y - r,
            x + r,
            y + r,
            fill="lightgray",
            outline="black",
            width=2
        )

        self.canvas.create_text(
            x,
            y,
            text=vertex
        )

        self.vertex_items[vertex] = circle

    def get_vertex_at(self, x, y):
        for vertex, (vx, vy) in self.graph.positions.items():

            distance_squared = (x - vx) ** 2 + (y - vy) ** 2

            if distance_squared <= self.vertex_radius ** 2:
                return vertex

        return None

    def select_vertex(self, vertex):

        if self.selected_vertex is None:

            self.selected_vertex = vertex

            self.highlight_vertex(vertex)

        else:

            if vertex != self.selected_vertex:
                self.create_edge(
                    self.selected_vertex,
                    vertex
                )

            self.unhighlight_vertex(self.selected_vertex)

            self.selected_vertex = None

    def highlight_vertex(self, vertex):
        self.canvas.itemconfigure(
            self.vertex_items[vertex],
            outline="blue",
            width=3
        )

    def unhighlight_vertex(self, vertex):
        self.canvas.itemconfigure(
            self.vertex_items[vertex],
            outline="black",
            width=2
        )

    def create_edge(self, vertex1, vertex2):

        if (vertex1, vertex2) in self.graph.edges:
            return

        if (vertex2, vertex1) in self.graph.edges:
            return

        self.graph.add_edge(vertex1, vertex2)

        x1, y1 = self.graph.positions[vertex1]
        x2, y2 = self.graph.positions[vertex2]

        self.canvas.create_line(
            x1,
            y1,
            x2,
            y2,
            fill="black",
            width=2
        )

        # Move the edge behind the vertices
        self.canvas.tag_lower(self.canvas.find_all()[-1])

    def is_position_occupied(self, x, y):
        for vx, vy in self.graph.positions.values():
            distance_squared = (x - vx) ** 2 + (y - vy) ** 2

            if distance_squared < self.min_vertex_distance ** 2:
                return True

        return False

    def color_vertex(self, vertex, color):
        color_map = {
            "r": "red",
            "g": "green",
            "b": "blue",
            "y": "yellow",
            "o": "orange" # Only used for 5 color solution, not needed for 4-coloring 
        }

        self.canvas.itemconfigure(
            self.vertex_items[vertex],
            fill=color_map[color]
        )

    def reset_vertex_colors(self):
        for vertex in self.graph.vertices:
            self.canvas.itemconfigure(
                self.vertex_items[vertex],
                fill="lightgray"
            )

    def solve(self):
        self.reset_vertex_colors()
        coloring = self.solver(self.graph, colors=['r', 'g', 'b', 'y'])

        if coloring is None:
            print("Graph is not 4-colorable.")

            if self.attempt_5_colors:
                print("Attempting to solve with 5 colors...")
                coloring = self.solver(self.graph, colors=['r', 'g', 'b', 'y', 'o'])
                
                if coloring is not None:
                    print("Graph is 5-colorable.")
                    print(coloring)
                    for vertex, color in coloring.items():
                        self.color_vertex(vertex, color)
                    messagebox.showinfo(
                        "5-Coloring",
                        "The graph is not 4-colorable, but it is 5-colorable."
                    )
                    return

                messagebox.showinfo(
                    "5-Coloring",
                    "The graph is not 5-colorable."
                )
                
                return

            messagebox.showinfo(
                "4-Coloring",
                "The graph is not 4-colorable."
            )

            return

        print("Graph is 4-colorable.")
        print(coloring)

        for vertex, color in coloring.items():
            self.color_vertex(vertex, color)


    def clear_graph(self):
        self.canvas.delete("all")

        self.graph.clear()

        self.next_vertex_id = 0
        self.selected_vertex = None

    def run(self):
        self.root.mainloop()

## App

The user can create nodes and edges using the UI. After creating a graph, the user can check if the graph is four colorable or not. If it is four colorable, the nodes will be colored accordingly. Once the nodes are colored, the graph can extended and colored again.

In [45]:
four_color_app = GraphGUI(solve_graph)
four_color_app.run()

Solution found:
[V_e_b = False,
 V_b_y = False,
 V_c_g = False,
 V_b_b = False,
 V_c_b = True,
 V_a_y = True,
 V_b_g = True,
 V_d_r = True,
 V_a_r = False,
 V_e_r = True,
 V_e_y = False,
 V_a_g = False,
 V_e_g = False,
 V_b_r = False,
 V_c_r = False,
 V_a_b = False,
 V_d_g = False,
 V_d_y = False,
 V_d_b = False,
 V_c_y = False]
Graph is 4-colorable.
{'b': 'g', 'c': 'b', 'd': 'r', 'a': 'y', 'e': 'r'}
Solution found:
[V_d_b = False,
 V_e_b = False,
 V_b_y = True,
 V_c_g = True,
 V_b_b = False,
 V_c_b = False,
 V_f_r = False,
 V_b_g = False,
 V_d_r = True,
 V_f_y = False,
 V_a_y = False,
 V_a_r = False,
 V_e_r = True,
 V_a_g = False,
 V_e_y = False,
 V_e_g = False,
 V_b_r = False,
 V_c_r = False,
 V_a_b = True,
 V_f_g = False,
 V_d_g = False,
 V_c_y = False,
 V_f_b = True,
 V_d_y = False]
Graph is 4-colorable.
{'f': 'b', 'b': 'y', 'c': 'g', 'd': 'r', 'a': 'b', 'e': 'r'}
Graph is not 4-colorable.


## Trivial Solution

Removing the condition that each vertex must contain at least one color will generate the trivial solution.

In [46]:
def trivial_solve_graph(graph, colors = None):
    if colors is None:
        colors = ['r', 'g', 'b', 'y']  # Default colors

    s = Solver()

    # Define boolean variables for each vertex and color

    vertex_color_variables = {}
    vertices = graph.vertices
    edges = graph.edges

    for v in vertices:
        for c in colors:
            vertex_color_variables[(v, c)] = Bool(f"V_{v}_{c}")

    for v in vertices:
        # Removed the constraint that each vertex must have at least one color to make it trivial
        
        # s.add(
        #     Or(
        #         *(vertex_color_variables[(v, c)] for c in colors)
        #     )
        # ) # Each vertex has at least one color


        s.add(
            And(
                *(
                    Or(
                        Not(vertex_color_variables[(v, c1)]), 
                        Not(vertex_color_variables[(v, c2)])
                    ) 
                    for i, c1 in enumerate(colors) 
                    for j, c2 in enumerate(colors) 
                    if i < j
                )
            )
        ) # Each vertex has only one color


    for u, v in edges:
        s.add(
            And(
                *(
                    Or(
                        Not(vertex_color_variables[(u, c)]), 
                        Not(vertex_color_variables[(v, c)])
                    ) for c in colors
                )
            )
        ) # connected nodes do not have the same color


    if s.check() == sat:
        model = s.model()
        print("Solution found:")
        print(s.model())
        return {
            v: c for v in graph.vertices 
            for c in colors 
            if is_true(model.evaluate(vertex_color_variables[(v, c)]))
        }
    else:
        return None

In [47]:
trivial_color_app = GraphGUI(trivial_solve_graph)
trivial_color_app.run()

Solution found:
[V_b_g = False,
 V_b_y = False,
 V_a_g = False,
 V_c_g = False,
 V_c_b = False,
 V_b_r = False,
 V_c_r = False,
 V_b_b = False,
 V_a_b = False,
 V_c_y = False,
 V_a_y = False,
 V_a_r = False]
Graph is 4-colorable.
{}
Solution found:
[V_b_g = False,
 V_b_y = False,
 V_c_g = False,
 V_a_g = False,
 V_c_b = False,
 V_b_r = False,
 V_c_r = False,
 V_b_b = False,
 V_a_b = False,
 V_c_y = False,
 V_a_y = False,
 V_a_r = False]
Graph is 4-colorable.
{}
Solution found:
[V_b_g = False,
 V_b_y = False,
 V_a_g = False,
 V_c_g = False,
 V_c_b = False,
 V_b_r = False,
 V_c_r = False,
 V_b_b = False,
 V_a_b = False,
 V_c_y = False,
 V_a_y = False,
 V_a_r = False]
Graph is 4-colorable.
{}
Solution found:
[V_b_g = False,
 V_b_y = False,
 V_c_g = False,
 V_a_g = False,
 V_c_b = False,
 V_b_r = False,
 V_c_r = False,
 V_b_b = False,
 V_a_b = False,
 V_c_y = False,
 V_a_y = False,
 V_a_r = False]
Graph is 4-colorable.
{}
Solution found:
[V_b_g = False,
 V_b_y = False,
 V_a_g = False,
 V_c

## 5-Colorable

If the graph is nonplaner and can not be colored with 4 colors, it will automatically attempt to color the graph using 5 colors.

In [49]:
five_color_app = GraphGUI(solve_graph, attempt_5_colors=True)
five_color_app.run()

Graph is not 4-colorable.
Attempting to solve with 5 colors...
Solution found:
[V_e_b = False,
 V_b_y = False,
 V_c_g = False,
 V_b_b = False,
 V_c_b = False,
 V_d_o = False,
 V_a_o = False,
 V_c_o = True,
 V_b_g = True,
 V_d_r = False,
 V_a_r = False,
 V_a_y = False,
 V_e_r = True,
 V_e_y = False,
 V_a_g = False,
 V_e_o = False,
 V_b_o = False,
 V_b_r = False,
 V_c_r = False,
 V_e_g = False,
 V_a_b = True,
 V_d_g = False,
 V_c_y = False,
 V_d_y = True,
 V_d_b = False]
Graph is 5-colorable.
{'b': 'g', 'c': 'o', 'd': 'y', 'a': 'b', 'e': 'r'}
Graph is not 4-colorable.
Attempting to solve with 5 colors...
Solution found:
[V_b_y = False,
 V_d_o = False,
 V_f_r = False,
 V_c_o = False,
 V_a_o = False,
 V_e_y = False,
 V_b_o = False,
 V_a_b = True,
 V_f_o = True,
 V_d_g = False,
 V_f_g = False,
 V_d_b = False,
 V_c_y = False,
 V_f_b = False,
 V_c_g = False,
 V_b_b = False,
 V_c_b = False,
 V_b_g = True,
 V_d_r = False,
 V_f_y = False,
 V_a_r = False,
 V_a_y = False,
 V_e_r = False,
 V_a_g = 